# Learned Bloom Filters — 4. The Learned Filter Designs

Three designs, each guaranteeing **zero false negatives**:

1. **Learned BF (LBF)** — oracle with threshold τ. If `f(x) ≥ τ` say "in S". For members
   with `f(x) < τ` (the oracle's misses) build a **backup Bloom filter** so we never miss
   them. `FPR = a + (1-a)·b`, where `a` = oracle FP-rate at τ, `b` = backup FPR.
2. **Sandwiched LBF** (Mitzenmacher 2018) — add an initial Bloom filter *before* the oracle
   to remove most non-members first. Plain LBF is the special case `f0 = 1.0`.
3. **Partitioned LBF / PLBF** (Vaidya 2021) — split the score range into regions, each with
   its own backup FPR (a water-filling allocation).

We score the oracle once, then search for the cheapest configuration at each target FPR.

In [1]:
# Download the URL dataset (~195k URLs, labelled malicious=1 / benign=0)
import os
import ssl
import urllib.request
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/incertum/cyber-matrix-ai/"
    "master/Malicious-URL-Detection-Deep-Learning/data/url_data_mega_deep_learning.csv"
)

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/url_data.csv"):
    # Relax TLS just for this one download (some Colab images lag on CA certs)
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120, context=ctx) as response:
        with open("data/url_data.csv", "wb") as out:
            out.write(response.read())

print("Dataset ready.")

Dataset ready.


In [2]:
import numpy as np
import pandas as pd


def load_and_split(seed=42):
    """Set S = all malicious URLs (the things we store).
    The benign URLs are split into three groups:
      - train : teach the oracle what 'safe' looks like
      - val   : pick the threshold / settings
      - test  : untouched, used only to report the final false-positive rate
    """
    df = pd.read_csv("data/url_data.csv")
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna().drop_duplicates("url").reset_index(drop=True)

    malicious = df[df.isMalicious == 1].url.tolist()
    benign = df[df.isMalicious == 0].url.sample(frac=1.0, random_state=seed).tolist()

    n = len(benign)
    neg_train = benign[: n // 2]
    neg_val = benign[n // 2 : 3 * n // 4]
    neg_test = benign[3 * n // 4 :]
    return malicious, neg_train, neg_val, neg_test

In [3]:
import math
import hashlib
import numpy as np


class BloomFilter:
    """A standard Bloom filter: m bits, k hash functions, never a false negative."""

    def __init__(self, m_bits, n_expected=None, k=None):
        self.m = int(max(1, m_bits))
        if k is None and n_expected:
            # Best number of hash functions for m bits and n keys is (m/n) * ln2
            k = round((self.m / n_expected) * math.log(2))
        self.k = int(max(1, k or 7))
        self.bits = np.zeros(self.m, dtype=np.uint8)

    def _positions(self, item):
        # Kirsch-Mitzenmacher trick: one MD5 -> two numbers -> k bit positions
        digest = hashlib.md5(item.encode("utf-8", "ignore")).digest()
        h1 = int.from_bytes(digest[:8], "little")
        h2 = int.from_bytes(digest[8:], "little") | 1
        return [(h1 + i * h2) % self.m for i in range(self.k)]

    def add(self, item):
        for p in self._positions(item):
            self.bits[p] = 1

    def add_all(self, items):
        for item in items:
            self.add(item)

    def __contains__(self, item):
        # All k bits set -> "probably in"; any bit is 0 -> "definitely not in"
        return all(self.bits[p] for p in self._positions(item))


def bf_theoretical_fpr(m, n, k):
    # Textbook false-positive rate of a Bloom filter
    return (1 - math.exp(-k * n / m)) ** k


def bf_bits_for_fpr(n, target_fpr):
    # Bits needed for n keys at a target FPR p:  m = -n*ln(p) / (ln2)^2
    if target_fpr <= 0:
        return 10 ** 9
    return math.ceil(-n * math.log(target_fpr) / (math.log(2) ** 2))

In [4]:
import re
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, vstack, csr_matrix

SUSPICIOUS = ["login", "secure", "account", "update", "verify", "bank",
              "confirm", "webscr", "signin", "paypal", "wp-", "admin",
              ".php", ".exe"]
SPECIAL = list("./-_?=&@%:#~+,;!$")


def lexical_features(url):
    """Hand-crafted numeric features describing one URL."""
    url = str(url)
    lower = url.lower()
    n = len(url) or 1

    # Shannon entropy = how 'random-looking' the characters are
    byte_vals = np.frombuffer(url.encode("utf-8", "ignore"), dtype=np.uint8)
    if byte_vals.size:
        _, counts = np.unique(byte_vals, return_counts=True)
        probs = counts / counts.sum()
        entropy = float(-(probs * np.log2(probs)).sum())
    else:
        entropy = 0.0

    feats = [
        len(url),
        len(lower.split("/")[0]),                          # host length
        sum(c.isdigit() for c in url) / n,
        sum(c.isalpha() for c in url) / n,
        entropy,
        url.count("."), url.count("/"), url.count("-"),
        url.count("="), url.count("?"), url.count("@"), url.count("%"),
        sum(lower.count(w) for w in SUSPICIOUS),           # suspicious keywords
        int(bool(re.search(r"\d+\.\d+\.\d+\.\d+", url))),   # contains an IP address?
        max((len(t) for t in re.split(r"[./\-_]", url)), default=0),
    ]
    feats += [url.count(c) / n for c in SPECIAL]           # ratio of each special char
    return feats


def train_oracle(malicious, neg_train, ngram_dim=4096):
    """Train the logistic-regression oracle. Returns (score_function, n_features).
    Features = lexical features + hashed character 3-to-5 grams (the hashing trick)."""
    hasher = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                               n_features=ngram_dim, alternate_sign=False, norm="l2")

    def featurize(urls):
        lexical = np.array([lexical_features(u) for u in urls], dtype=np.float32)
        ngrams = hasher.transform(urls)
        return lexical, ngrams

    lex_pos, ng_pos = featurize(malicious)
    lex_neg, ng_neg = featurize(neg_train)

    # Scale the lexical features so the linear model trains smoothly
    scaler = StandardScaler().fit(np.vstack([lex_pos, lex_neg]))
    X_pos = hstack([csr_matrix(scaler.transform(lex_pos)), ng_pos]).tocsr()
    X_neg = hstack([csr_matrix(scaler.transform(lex_neg)), ng_neg]).tocsr()

    X = vstack([X_pos, X_neg]).tocsr()
    y = np.r_[np.ones(X_pos.shape[0]), np.zeros(X_neg.shape[0])]
    model = LogisticRegression(max_iter=3000, C=8.0, solver="liblinear").fit(X, y)
    n_features = X.shape[1]

    def score(urls):
        lexical, ngrams = featurize(urls)
        Xq = hstack([csr_matrix(scaler.transform(lexical)), ngrams]).tocsr()
        return model.predict_proba(Xq)[:, 1]

    return score, n_features

In [5]:
import numpy as np


def optimize_sandwiched(scores_pos, scores_neg, oracle_bits, target_fpr,
                        f0_grid=None, n_tau=200):
    """Find the smallest (fewest bits) learned filter that hits target_fpr.
    f0 is the initial-filter FPR (f0 = 1.0 means 'no initial filter', i.e. plain LBF).
    Returns the best configuration, or None if the target can't be reached."""
    sp = np.sort(scores_pos)
    sn = np.sort(scores_neg)
    n_pos = len(sp)

    if f0_grid is None:
        f0_grid = np.concatenate([[1.0], np.geomspace(0.05, 0.9, 12)])
    taus = np.quantile(sp, np.linspace(0.0, 0.99, n_tau))

    best = None
    for f0 in f0_grid:
        initial_bits = bf_bits_for_fpr(n_pos, f0) if f0 < 1.0 else 0
        for tau in taus:
            a = float(np.mean(sn >= tau))           # oracle's own false-positive rate
            below = 1.0 - a                          # fraction of negatives below tau
            c = int(np.sum(sp < tau))                # members the oracle would miss

            if f0 * a >= target_fpr:                 # oracle alone already too loose
                continue
            # Backup FPR needed so the whole thing lands on target_fpr
            backup_fpr = (target_fpr / f0 - a) / below if below > 0 else 0.0
            if backup_fpr <= 0:
                continue

            if c > 0 and backup_fpr < 1:
                backup_bits = bf_bits_for_fpr(c, min(backup_fpr, 0.999))
            else:
                backup_bits = 0

            total = initial_bits + oracle_bits + backup_bits
            if best is None or total < best["bits"]:
                best = {"bits": total, "tau": float(tau), "f0": float(f0),
                        "b": float(min(backup_fpr, 1.0)), "c": c}
    return best


def plbf_bits(scores_pos, scores_neg, oracle_bits, target_fpr, n_regions=8):
    """Partitioned LBF: split the score range into regions, give each region its own
    backup FPR by water-filling, return the total bits."""
    sp = np.sort(scores_pos)
    sn = np.sort(scores_neg)

    edges = np.quantile(sp, np.linspace(0, 1, n_regions + 1))
    edges[0] = -np.inf
    edges[-1] = np.inf

    g = np.histogram(sp, bins=edges)[0] / len(sp)        # member density per region
    h = np.histogram(sn, bins=edges)[0] / len(sn)        # non-member density per region
    ratio = np.where(h > 0, g / np.maximum(h, 1e-12), np.inf)

    # Water-filling: region FPR f_i = min(1, t * g_i / h_i); tune t so total FPR = target
    lo, hi = 0.0, 1e6
    for _ in range(60):
        t = (lo + hi) / 2
        f = np.minimum(1.0, t * ratio)
        if float(np.sum(h * f)) > target_fpr:
            hi = t
        else:
            lo = t
    f = np.minimum(1.0, lo * ratio)

    member_counts = np.histogram(sp, bins=edges)[0]
    bits = 0
    for count, region_fpr in zip(member_counts, f):
        if count > 0 and region_fpr < 1.0:
            bits += bf_bits_for_fpr(int(count), float(max(region_fpr, 1e-6)))
    return oracle_bits + bits

Train the oracle, then build the memory–FPR table for all four designs. (Configs are chosen on the validation negatives.)

In [6]:
malicious, neg_train, neg_val, neg_test = load_and_split()
score, n_features = train_oracle(malicious, neg_train)
oracle_bits = (n_features + 1) * 8

scores_malicious = score(malicious)
scores_val = score(neg_val)

targets = [3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4]


def to_kb(bits):
    return bits / 8 / 1024


def show(value):
    return "%5.1f" % value if value is not None else "  -  "


print(" target |   BF  |  LBF  | Sand. | PLBF  | best saving")
for F in targets:
    bf_bits = bf_bits_for_fpr(len(malicious), F)
    lbf = optimize_sandwiched(scores_malicious, scores_val, oracle_bits, F, f0_grid=np.array([1.0]))
    sand = optimize_sandwiched(scores_malicious, scores_val, oracle_bits, F)
    plbf = plbf_bits(scores_malicious, scores_val, oracle_bits, F)

    lbf_kb = to_kb(lbf["bits"]) if lbf else None
    sand_kb = to_kb(sand["bits"]) if sand else None
    plbf_kb = to_kb(plbf)

    options = [x for x in [lbf_kb, sand_kb, plbf_kb] if x is not None]
    saving = (1 - min(options) / to_kb(bf_bits)) * 100
    print(" %.0e | %5.1f | %s | %s | %s | %4.1f%%"
          % (F, to_kb(bf_bits), show(lbf_kb), show(sand_kb), show(plbf_kb), saving))

 target |   BF  |  LBF  | Sand. | PLBF  | best saving


 3e-02 |  65.4 |  28.4 |  28.4 |  21.6 | 67.0%


 1e-02 |  85.8 |  52.7 |  48.7 |  36.3 | 57.7%


 3e-03 | 108.3 |  89.8 |  71.2 |  58.0 | 46.5%


 1e-03 | 128.8 | 136.4 |  92.1 |  78.4 | 39.1%


 3e-04 | 151.2 |   -   | 123.0 | 100.9 | 33.3%


 1e-04 | 171.7 |   -   | 162.9 | 121.4 | 29.3%


### Build the real structure and check it on the held-out test set
A table of numbers proves nothing if the math is wrong, so we actually build the Sandwiched filter at 0.1% and query the **held-out test negatives** (never used for training or tuning).

In [7]:
target = 1e-3
config = optimize_sandwiched(scores_malicious, scores_val, oracle_bits, target)
tau = config["tau"]
f0 = config["f0"]
b = config["b"]

# Backup filter stores exactly the members the oracle would miss
missed = [u for u, s in zip(malicious, scores_malicious) if s < tau]
backup = BloomFilter(bf_bits_for_fpr(len(missed), min(b, 0.999)), n_expected=len(missed))
backup.add_all(missed)

# Sandwich: an initial filter in front (only if f0 < 1)
initial = None
if f0 < 1.0:
    initial = BloomFilter(bf_bits_for_fpr(len(malicious), f0), n_expected=len(malicious))
    initial.add_all(malicious)


def is_member(urls, scores):
    answers = []
    for url, s in zip(urls, scores):
        if initial is not None and url not in initial:
            answers.append(False)            # initial filter: definitely not in S
        elif s >= tau:
            answers.append(True)             # oracle is confident
        else:
            answers.append(url in backup)    # fall back to the safety net
    return np.array(answers)


scores_test = score(neg_test)
false_neg = int((~is_member(malicious, scores_malicious)).sum())
test_fpr = float(is_member(neg_test, scores_test).mean())

print("Sandwiched filter @ target %.0e" % target)
print("  false negatives           =", false_neg, " (must be 0)")
print("  FPR on held-out test set  = %.4f" % test_fpr)
print("  backup stores %d of %d members" % (len(missed), len(malicious)))

Sandwiched filter @ target 1e-03
  false negatives           = 0  (must be 0)
  FPR on held-out test set  = 0.0005
  backup stores 23725 of 73367 members


**Takeaway.** PLBF wins everywhere. **Plain LBF actually becomes *worse* than a standard Bloom filter at low FPR** (and infeasible below ~3e-4) — the known limitation that motivates the Sandwiched and Partitioned designs. The real-structure check confirms FN = 0 and the held-out FPR matches the budget, so the memory numbers are real.